# Template application

In [ ]:
template_file = 'entity.templ.html'

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
from lxml import etree
import IPython

# valid xml namespaces and schema for Confluence 6 storage format
# See 
xml_namespaces=[
    'xmlns="http://www.w3.org/1999/xhtml"',
    'xmlns:ac="http://www.atlassian.com/schema/confluence/4/ac/"',
    'xmlns:ri="http://www.atlassian.com/schema/confluence/4/ri/"',
    'xmlns:acxhtml="http://www.atlassian.com/schema/confluence/4/"'
]

def encapsulate_storage_format(xml):
    return '<?xml version="1.0"?><root doc="container to properly encapsulate xml" {} >\n{}\n</root>'.format(
            ' '.join(xml_namespaces), xml)

def beautify_xml(flat_xml):
    try:
        encapsulated = encapsulate_storage_format(flat_xml)
        dom = xml.dom.minidom.parseString(encapsulated)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        log.error('Expat error {}'.format(e))
        raise Exception(e)
    
def validate_storage_format(xml_in_storage_format):
        '''Validate if input xml conforms to Confluence storage format specifications'''
        parser = etree.XMLParser(dtd_validation=False)
        try:
            etree.fromstring(encapsulate_storage_format(xml_in_storage_format), parser)
            return None
        except xml.parsers.expat.ExpatError as e:
            m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
            message = 'Malformed xml ' + flat_xml[:50] + ' ...'
            if m:
                lines = wrapped.splitlines()
                messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
            else:
                message = message + flat_xml[:50] + ' ...'
            return message

beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('all fine!')

In [ ]:
from xml.sax.saxutils import escape


class Toolbox:
    '''
    The toolbox contains mapping information of IM elements
    '''
    
    def __init__(self, data, confluence, language='de'):
        self.language = language
        # dictionary with key = element_key ('E233322', 'A132452', ...)
        self.json_data = data
        self.content_map = {}
        self.confluence = confluence
    
    def translate(self, field):
        return escape(field[self.language])
    
    def href(self, element_key):
        '''Returns the url of an element'''
        return str(element_key)
    
    def page(self, element_key):
        '''Returns the page object of an element or None if there is no page yet
        A page object is a dictionary containing 'pageid' and 'name' 
        '''
        return pagemap.get(element_key)
        
    def register_page_id(self, element_key, page_id):
        element = content_map.get(element_key, {})
        element['pageid'] = page_id
    
    

toolbox = Toolbox(None, None, 'de')

# Load datasource

In [ ]:
import json

data = None
with open('testdata/IM-sample.json', 'r') as source:
     data = json.load(source)

entities = data['entities']
# Print some information on what was loaded
print('Model "{}" contains {} entities:'.format(data['model']['name'], len(entities)))
list(map(lambda e: (e, toolbox.translate(entities[e]['name'])), data['entities']))

In [ ]:
first_entity_name = 'ENTI12701'

first = data['entities'][first_entity_name]
log.warning('Working with entity ' + first_entity_name + ". Name: " + Toolbox('de').translate(first['name']))
first

In [ ]:
entity_template = None
with open('./templates/' + template_file, 'r') as f:
    entity_template = f.read()

assert entity_template

IPython.display.Code(entity_template)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)

template = env.get_template(template_file)
rendered_template = template.render(entity=first, util=Toolbox('de'))
content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_template) # strip comment lines
IPython.display.Code(content_xml)

In [ ]:
IPython.display.HTML(content_xml)

In [ ]:
validate_storage_format(content_xml)